In [1]:
# Import necessary packages
import torch
from torch import nn
import torchinfo
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import time


import brevitas.nn as qnn
from brevitas.quant import Int8Bias
from brevitas.inject.enum import ScalingImplType
from brevitas.inject.defaults import Int8ActPerTensorFloatMinMaxInit

# Prepare data loader
from torch.utils.data import Dataset, DataLoader
import h5py
from sklearn.metrics import accuracy_score

In [2]:
# Make sure that CUDA is available - without cuda we would be attempting to run the model on CPU which is tooooo slow. 
assert torch.cuda.is_available(), 'Cuda not available'

In [3]:
def quantize(data):
    _min=-55.0
    _max=55.0
    _range=_max-_min
    # Data in 0 to 1 
    quant_data = ((data - _min)/ _range) 

    # Data in -128 to 127
    quant_data = (quant_data * 255.0) - 128.0
    quant_data = np.round(quant_data)
    quant_data = np.clip(quant_data,-128.0, 127.0)
    return quant_data.astype(np.int8)

class radio_dataset_spec(Dataset):
    all_IQ=[]
    all_mod=[]
    all_snr=[]
    train_sampler:torch.utils.data.SubsetRandomSampler
    val_sampler:torch.utils.data.SubsetRandomSampler
    test_sampler:torch.utils.data.SubsetRandomSampler
    def __init__(self,dataset_path:str,fpmsc:int=4096):
        iq_key="all_specs_FP32"
        mod_key="all_labels"
        snr_key="all_SNRs"
        
        np.random.seed(2021) #maintain deterministic randomness
        print(f"loading {dataset_path}")
        h5_file = h5py.File(dataset_path,'r')
        print(f"All available keys: {list(h5_file.keys())}")
        self.all_IQ = h5_file[iq_key]
        self.all_mod = h5_file[mod_key][:,0].flatten()
        self.all_snr = h5_file[snr_key][:,0].flatten()
        # print(f"Raw values shape: {self.all_IQ.shape}")
        # print(f"Raw values min/max range: {np.min(self.all_IQ)}, {np.max(self.all_IQ)}, {self.all_IQ.dtype}")
        # print(f"Labels shape: {self.all_mod.shape}")
        # print(f"SNR shape: {self.all_snr.shape}")
        
        train_indices = []
        test_indices = []
        val_indices = []
        # fpmsc: frames per mod snr combination, the dataset is organized as mod major, snr minor
        # each mod-snr-combination has fpsmc frames, we split those fpsmc frames uniformly to train, test, val indices
        for i in range(0,len(self.all_IQ),fpmsc):
            indices_subclass=list(range(i,i+fpmsc))
            split = int(np.ceil(0.8 * fpmsc)) #split 80,10,10
            split2 = int(np.ceil(0.9 * fpmsc))
            
            np.random.shuffle(indices_subclass)
            
            train_indices_subclass = indices_subclass[:split]
            val_indices_subclass = indices_subclass[split:split2]
            test_indices_subclass = indices_subclass[split2:]
            
            train_indices.extend(train_indices_subclass)
            test_indices.extend(test_indices_subclass)
            val_indices.extend(val_indices_subclass) 
        
        self.train_sampler = torch.utils.data.SubsetRandomSampler(train_indices)
        self.val_sampler = torch.utils.data.SubsetRandomSampler(val_indices)
        self.test_sampler = torch.utils.data.SubsetRandomSampler(test_indices)
        
        # print('Training set size: ',len(self.train_sampler))
        # print('Val set size: ',len(self.val_sampler))
        # print('Test set size: ',len(self.test_sampler))
    
    def __getitem__(self, idx):
        # transpose frame into Pytorch channels-first format (NCL = -1,2,1024)
        x = self.all_IQ[idx]      # (64, 32)
        x = quantize(x)
        x = torch.tensor(x, dtype=torch.float32)
        # swap dimensions so channels=32, time=64
        x = x.transpose(0, 1)
        return x, self.all_mod[idx], self.all_snr[idx]

In [4]:
def quantized_vgg10(model_body_bit:int)->nn.Sequential:
    a_bits = model_body_bit  # a_bits is the bit width for ReLu
    w_bits = model_body_bit # w_bits is the bit width for all the weights
    
    input_bits:int=8
    new_min=-128.0
    new_max=128
    filters_conv = 64
    filters_dense = 128
    
    class InputQuantizer(Int8ActPerTensorFloatMinMaxInit):
        #Quantize to input_bits
        bit_width = input_bits
        
        #Min max value of the input. Set to the range value of the input 
        min_val = new_min #the min value of the input(dataset) before going through the model
        max_val = new_max #the max value of the input(dataset) before going through the model
        scaling_impl_type = ScalingImplType.CONST # Fix the quantization range to [min_val, max_val]
    
    return nn.Sequential(
            # Input quantization
            qnn.QuantHardTanh(act_quant=InputQuantizer),
        
            qnn.QuantConv1d(32, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(64),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(64, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(64),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(64, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(64),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
            
            qnn.QuantConv1d(64, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(64),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            qnn.QuantConv1d(64, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(64),
            qnn.QuantReLU(bit_width=a_bits),
            nn.MaxPool1d(2),
        
            # qnn.QuantConv1d(64, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            # nn.BatchNorm1d(64),
            # qnn.QuantReLU(bit_width=a_bits),
            # nn.MaxPool1d(2),
        
            # qnn.QuantConv1d(64, 64, 3, padding=1, weight_bit_width=w_bits, bias=False),
            # nn.BatchNorm1d(64),
            # qnn.QuantReLU(bit_width=a_bits),
            # nn.MaxPool1d(2),
            
            nn.Flatten(),
        
            qnn.QuantLinear(128, 128, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(128),
            qnn.QuantReLU(bit_width=a_bits),
        
            qnn.QuantLinear(128, 128, weight_bit_width=w_bits, bias=False),
            nn.BatchNorm1d(128),
            qnn.QuantReLU(bit_width=a_bits, return_quant_tensor=True),
        
            qnn.QuantLinear(128, 15, weight_bit_width=w_bits, bias=True, bias_quant=Int8Bias),
        )

In [5]:
from src.test_result import *
def get_model_testing_result(model, dataset:radio_dataset_spec, batch_size:int=1024)->testing_result:   
    model.to('cuda')
    # print(torchinfo.summary(model,input_size=(1,2,1024),depth=1));
    test_dataset=DataLoader(dataset, batch_size=batch_size, sampler=dataset.test_sampler)
    y_pred = np.empty((0)) 
    y_exp = np.empty((0))
    y_snr = np.empty((0))
    model.eval()

    with torch.no_grad():
        for data in tqdm(test_dataset, desc="Batches"):
            inputs, target, snr = data
            inputs = inputs.to('cuda').float()
            output = model(inputs).cpu()
            #model raw output 15 mods prediction values, we can do the softmax here
            output = np.argmax(output,axis=1)
            y_pred = np.concatenate((y_pred,output))
            y_exp = np.concatenate((y_exp,target))
            y_snr = np.concatenate((y_snr,snr))

    return testing_result(y_pred,y_exp,y_snr,use_snr=True)

## Main

In [11]:
mod_classes = ["BPSK", 
                "QPSK", 
                "8PSK",
                "16QAM",
                "32QAM", 
                "64QAM", 
                "128QAM", 
                "256QAM",
                "16APSK", 
                "32APSK", 
                "64APSK", 
                "128APSK",
                "FM", 
                "AM-DSB-SC", 
                "AM-SSB-SC"]

lbl_count:int=len(mod_classes)
snr_classes=np.arange(0.0,31.0,2.0)
plot_parent_dir="cross_testing_result"
Path(plot_parent_dir).mkdir(exist_ok=True)

       
def main():    
    dataset=radio_dataset_spec("datasets/spectrogram/spectrogram_matgen_rayleigh_ppm0_20250928_64x32.h5")
    model=quantized_vgg10(8)
    model.load_state_dict(torch.load("archive/spectrogram/rayleigh_matgen_ppm20_spectrogram/8bit/model_brevitas.pth"))
    model.to('cuda')
    result=get_model_testing_result(model,dataset)
    
    file_name=f"./result/GPU_SPEC_trained[8b_ray_ppm20]_test[ray_ppm0].npz"
    assert not os.path.isfile(file_name), f"file {file_name} exist, avoid override"
    result.save_file(file_name)
    print(f"saved to {file_name}")
                
main()

loading datasets/spectrogram/spectrogram_matgen_rayleigh_ppm0_20250928_64x32.h5
All available keys: ['all_SNRs', 'all_labels', 'all_specs_FP32']


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

saved to ./result/GPU_SPEC_trained[8b_ray_ppm20]_test[ray_ppm0].npz
